# MarketPulse: Multimodal S&P 500 Next-Day Movement Predictor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

Fuses **FinBERT** news sentiment with a **TCN** price encoder via a learned attention gate
to predict next-day directional movement of S&P 500 stocks.

**How to run on Colab (T4):**
1. `Runtime` -> `Change runtime type` -> select **T4 GPU**.
2. `Runtime` -> `Run all`.
3. (Optional) Set `MOUNT_DRIVE = True` in the setup cell to persist data/checkpoints to your Drive across sessions.
4. Set `SMOKE_TEST = True` in the **Config** cell for a 5-ticker / 1-epoch dry run (~5 min). Default is `False` (full spec).

The notebook trains 5 models and produces:
- `results/metrics.json` with accuracy / MCC / Sharpe
- `results/figures/*.png` (5 plots) and `metrics_summary_table.csv`

## 1. Setup: runtime check, optional Drive mount, dependencies

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    try:
        out = subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT).decode()
        print(out.split("\n")[0:15])
        if "Tesla T4" in out or "T4" in out:
            print("Confirmed: T4 GPU detected.")
        else:
            print("WARNING: GPU detected but it does not look like a T4. Notebook will still work.")
    except Exception as e:
        print("=" * 70)
        print("WARNING: nvidia-smi failed. You are NOT on a GPU runtime.")
        print("Go to Runtime -> Change runtime type -> T4 GPU, then re-run.")
        print("=" * 70)
        print(e)
else:
    print("Not on Colab \u2014 will use local CUDA/MPS/CPU autodetect.")

In [ ]:
MOUNT_DRIVE = False

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/MarketPulse"
elif IN_COLAB:
    BASE_DIR = "/content/MarketPulse"
else:
    BASE_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(BASE_DIR, "data")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
GDELT_CACHE_DIR = os.path.join(BASE_DIR, ".gdelt_cache")

for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, GDELT_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"BASE_DIR = {BASE_DIR}")
for name, d in [("data", DATA_DIR), ("checkpoints", CHECKPOINT_DIR),
                ("results", RESULTS_DIR), ("figures", FIGURES_DIR), ("gdelt_cache", GDELT_CACHE_DIR)]:
    print(f"  {name:12s} -> {d}")

In [ ]:
%pip install -q torch transformers yfinance pandas scikit-learn matplotlib pyarrow shap tqdm alpaca-py

In [ ]:
import json
import math
import time
import random
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
try:
    from transformers import set_seed as _hf_set_seed
    _hf_set_seed(SEED)
except Exception:
    pass

## 2. Config

All hyperparameters live here. Flip `SMOKE_TEST = True` for a fast end-to-end sanity run.

In [ ]:
SMOKE_TEST = False

ALPACA_API_KEY = "your_key_here"
ALPACA_SECRET_KEY = "your_secret_here"

STOCKS_FULL = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "GOOG", "TSLA", "FB", "BRK-B", "JPM", "JNJ",
    "V",    "PG",   "UNH",  "NVDA",  "HD",   "MA",   "BAC", "DIS",   "PYPL", "ADBE",
    "CMCSA","XOM",  "VZ",   "NFLX",  "INTC", "T",    "CSCO","KO",    "PFE",  "MRK",
    "PEP",  "ABT",  "WMT",  "CRM",   "TMO",  "ABBV", "CVX", "ACN",   "AVGO", "COST",
    "MDT",  "NKE",  "MCD",  "DHR",   "WFC",  "NEE",  "TXN", "LLY",   "LIN",  "HON",
    "QCOM", "UNP",  "PM",   "BMY",   "C",    "ORCL", "AMGN","UPS",   "MS",   "BA",
    "LOW",  "RTX",  "SBUX", "IBM",   "CAT",  "BLK",  "GS",  "AMD",   "AMAT", "INTU",
    "GE",   "DE",   "AXP",  "MMM",   "SPGI", "GILD", "TGT", "ISRG",  "PLD",  "BKNG",
    "CHTR", "MO",   "CVS",  "ANTM",  "NOW",  "SCHW", "ZTS", "MDLZ",  "CI",   "TJX",
    "SYK",  "ADP",  "CB",   "DUK",   "USB",  "EL",   "MU",  "PNC",   "BDX",  "FIS",
]

if SMOKE_TEST:
    STOCKS = ["AAPL", "MSFT", "AMZN", "GOOGL", "TSLA"]
    DATE_TRAIN_START = "2022-07-01"
    DATE_TRAIN_END   = "2022-12-31"
    DATE_VAL_START   = "2023-01-01"
    DATE_VAL_END     = "2023-01-31"
    DATE_TEST_START  = "2023-02-01"
    DATE_TEST_END    = "2023-03-31"
    BATCH_SIZE       = 16
    MAX_EPOCHS       = 1
    EARLY_STOPPING_PATIENCE = 1
else:
    STOCKS = STOCKS_FULL
    DATE_TRAIN_START = "2021-01-01"
    DATE_TRAIN_END   = "2022-12-31"
    DATE_VAL_START   = "2023-01-01"
    DATE_VAL_END     = "2023-03-31"
    DATE_TEST_START  = "2023-04-01"
    DATE_TEST_END    = "2023-12-31"
    BATCH_SIZE       = 64
    MAX_EPOCHS       = 15
    EARLY_STOPPING_PATIENCE = 3

DOWNLOAD_START = DATE_TRAIN_START
DOWNLOAD_END   = DATE_TEST_END

WINDOW = 5
LR_FINBERT = 2e-5
LR_TCN = 1e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.4
TCN_CHANNELS = 64
TCN_LAYERS = 4
EMBED_DIM = 256
HIGH_NEWS_THRESHOLD = 5
PRICE_FEATURES = ["open", "high", "low", "close", "volume", "return"]

EXPECTED_ACCURACY = {
    "tfidf_lr":            0.52,
    "tcn_only":            0.54,
    "finbert_only":        0.57,
    "marketpulse_nogate":  0.58,
    "marketpulse":         0.60,
}

print(f"SMOKE_TEST  = {SMOKE_TEST}")
print(f"STOCKS      = {len(STOCKS)} tickers")
print(f"Train       = {DATE_TRAIN_START} -> {DATE_TRAIN_END}")
print(f"Val         = {DATE_VAL_START} -> {DATE_VAL_END}")
print(f"Test        = {DATE_TEST_START} -> {DATE_TEST_END}")
print(f"Batch size  = {BATCH_SIZE}, Max epochs = {MAX_EPOCHS}")

## 3. Data download: prices (yfinance)

Downloads OHLCV for all tickers, drops tickers with >10% missing days, computes the daily return, and z-score-normalizes each feature on a **causal rolling 252-day window** (no lookahead bias). Cached to `data/prices.parquet`.

In [ ]:
import yfinance as yf

PRICES_PATH = os.path.join(DATA_DIR, "prices.parquet")

def _causal_zscore(s: pd.Series, window: int = 252) -> pd.Series:
    mu  = s.shift(1).rolling(window, min_periods=20).mean()
    sig = s.shift(1).rolling(window, min_periods=20).std().replace(0, np.nan)
    return ((s - mu) / sig).fillna(0.0)

def download_prices(tickers: List[str], start: str, end: str, out_path: str) -> pd.DataFrame:
    if os.path.exists(out_path):
        print(f"[prices] cached -> {out_path}")
        return pd.read_parquet(out_path)

    print(f"[prices] downloading {len(tickers)} tickers from {start} to {end} ...")
    raw = yf.download(
        tickers, start=start, end=end,
        group_by="ticker", auto_adjust=False, progress=False, threads=True,
    )

    rows = []
    expected_days = None
    for t in tickers:
        try:
            sub = raw[t].copy() if isinstance(raw.columns, pd.MultiIndex) else raw.copy()
        except Exception:
            continue
        sub = sub.dropna(how="all")
        if sub.empty:
            continue
        sub = sub.rename(columns={c: c.lower() for c in sub.columns})
        if not {"open", "high", "low", "close", "volume"}.issubset(sub.columns):
            continue
        sub = sub.reset_index().rename(columns={"Date": "date"})
        sub["ticker"] = t
        sub["return"] = sub["close"].pct_change().fillna(0.0)
        rows.append(sub[["ticker", "date"] + PRICE_FEATURES])
        if expected_days is None:
            expected_days = len(sub)

    df = pd.concat(rows, ignore_index=True)
    counts = df.groupby("ticker").size()
    keep = counts[counts >= 0.9 * counts.max()].index.tolist()
    dropped = sorted(set(df["ticker"].unique()) - set(keep))
    if dropped:
        print(f"[prices] dropped {len(dropped)} tickers with >10% missing: {dropped}")
    df = df[df["ticker"].isin(keep)].copy()

    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    for feat in PRICE_FEATURES:
        df[feat] = df.groupby("ticker", group_keys=False)[feat].apply(_causal_zscore)

    df.to_parquet(out_path, index=False)
    print(f"[prices] saved {len(df):,} rows for {df['ticker'].nunique()} tickers -> {out_path}")
    return df

prices_df = download_prices(STOCKS, DOWNLOAD_START, DOWNLOAD_END, PRICES_PATH)
prices_df.head()

## 4. Data download: news (Alpaca Markets News API)

Uses the [`alpaca-py`](https://alpaca.markets/docs/) `NewsClient` to fetch news per-ticker, paginating through `next_page_token` until exhausted. Requires a free Alpaca account ([sign up here](https://alpaca.markets)) \u2014 paste your **paper trading** keys into `ALPACA_API_KEY` / `ALPACA_SECRET_KEY` in the Config cell above.

To avoid re-downloading on every Colab session, the result is persisted to **Google Drive** at `/content/drive/MyDrive/marketpulse/news.parquet`. If that file already exists, the cell loads it and skips the API entirely.

For each `(ticker, date)`: concatenates same-day headlines with `[SEP]` and truncates to 512 BERT tokens. Pairs with no headlines store `"NO_NEWS"`. Output schema is unchanged so nothing downstream changes.

In [ ]:
from transformers import AutoTokenizer

NEWS_PATH = os.path.join(DATA_DIR, "news.parquet")
NEWS_DRIVE_DIR  = "/content/drive/MyDrive/marketpulse"
NEWS_DRIVE_PATH = os.path.join(NEWS_DRIVE_DIR, "news.parquet")
ALPACA_PAGE_LIMIT = 50

def _truncate_to_tokens(text: str, tokenizer, max_tokens: int = 512) -> str:
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_tokens)
    return tokenizer.decode(ids, skip_special_tokens=True)

def _ensure_drive_mounted() -> bool:
    if not IN_COLAB:
        return False
    if os.path.exists("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.exists("/content/drive/MyDrive")
    except Exception as e:
        print(f"[news] Drive mount failed: {e}")
        return False

def _persist_to_drive(df: pd.DataFrame) -> None:
    if not _ensure_drive_mounted():
        return
    try:
        os.makedirs(NEWS_DRIVE_DIR, exist_ok=True)
        df.to_parquet(NEWS_DRIVE_PATH, index=False)
        print(f"[news] persisted to Drive -> {NEWS_DRIVE_PATH}")
    except Exception as e:
        print(f"[news] Drive write failed ({e}) \u2014 continuing with local file only")

def _try_load_from_drive() -> Optional[pd.DataFrame]:
    if not _ensure_drive_mounted():
        return None
    if not os.path.exists(NEWS_DRIVE_PATH):
        return None
    try:
        df = pd.read_parquet(NEWS_DRIVE_PATH)
        print(f"[news] loaded cached news from Drive -> {NEWS_DRIVE_PATH}  ({len(df):,} rows)")
        return df
    except Exception as e:
        print(f"[news] Drive cache read failed ({e}) \u2014 will re-fetch from Alpaca")
        return None

def _extract_news_items(resp) -> List:
    items = getattr(resp, "news", None)
    if items is None:
        items = getattr(resp, "data", None)
    if isinstance(items, dict):
        flat = []
        for v in items.values():
            if isinstance(v, list):
                flat.extend(v)
        items = flat
    return items or []

def _fetch_alpaca_news_for_ticker(client, ticker: str, start_dt, end_dt,
                                  page_limit: int = ALPACA_PAGE_LIMIT) -> List[Tuple[pd.Timestamp, str]]:
    from alpaca.data.requests import NewsRequest
    rows: List[Tuple[pd.Timestamp, str]] = []
    page_token: Optional[str] = None
    pages = 0
    while True:
        kwargs = dict(symbols=ticker, start=start_dt, end=end_dt, limit=page_limit)
        if page_token:
            kwargs["page_token"] = page_token
        try:
            req = NewsRequest(**kwargs)
        except TypeError:
            req = NewsRequest(symbols=ticker, start=start_dt, end=end_dt, limit=page_limit)

        resp = client.get_news(req)
        pages += 1
        items = _extract_news_items(resp)
        for n in items:
            ts = (getattr(n, "created_at", None)
                  or getattr(n, "updated_at", None)
                  or (n.get("created_at") if isinstance(n, dict) else None))
            headline = (getattr(n, "headline", None)
                        or (n.get("headline") if isinstance(n, dict) else None))
            if not ts or not headline:
                continue
            try:
                d = pd.to_datetime(ts, utc=True).tz_localize(None).normalize()
            except Exception:
                continue
            rows.append((d, str(headline).strip()))

        page_token = (getattr(resp, "next_page_token", None)
                      or (resp.get("next_page_token") if isinstance(resp, dict) else None))
        if not page_token or pages >= 500:
            break
    return rows

def download_news(tickers: List[str], start: str, end: str, trading_dates: pd.DatetimeIndex, out_path: str) -> pd.DataFrame:
    cached = _try_load_from_drive()
    if cached is not None:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        cached.to_parquet(out_path, index=False)
        return cached

    if os.path.exists(out_path):
        print(f"[news] local cache hit -> {out_path}")
        df = pd.read_parquet(out_path)
        _persist_to_drive(df)
        return df

    if not ALPACA_API_KEY or ALPACA_API_KEY == "your_key_here" \
       or not ALPACA_SECRET_KEY or ALPACA_SECRET_KEY == "your_secret_here":
        raise RuntimeError(
            "ALPACA_API_KEY / ALPACA_SECRET_KEY are not set. Get a free account at "
            "https://alpaca.markets, generate Paper Trading keys, and paste them into "
            "the Config cell, then re-run."
        )

    from alpaca.data.historical import NewsClient
    client = NewsClient(api_key=ALPACA_API_KEY, secret_key=ALPACA_SECRET_KEY)

    start_dt = datetime.fromisoformat(start)
    end_dt   = datetime.fromisoformat(end)

    print(f"[news] fetching Alpaca news for {len(tickers)} tickers ({start} -> {end}) ...")
    print(f"[news] paginating with limit={ALPACA_PAGE_LIMIT}; will persist to {NEWS_DRIVE_PATH if IN_COLAB else out_path}")
    tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")

    all_rows: List[Tuple[str, pd.Timestamp, str]] = []
    t_start = time.time()
    for i, t in enumerate(tickers, 1):
        t0 = time.time()
        try:
            rows = _fetch_alpaca_news_for_ticker(client, t, start_dt, end_dt)
        except Exception as e:
            print(f"[news] {i:3d}/{len(tickers)} {t:6s} ERROR: {e}", flush=True)
            rows = []
        for d, h in rows:
            all_rows.append((t, d, h))
        elapsed_t = time.time() - t0
        elapsed_total = time.time() - t_start
        eta = (elapsed_total / i) * (len(tickers) - i)
        print(f"[news] {i:3d}/{len(tickers)} {t:6s} headlines={len(rows):5d}  "
              f"t={elapsed_t:5.1f}s  total={elapsed_total/60:5.1f}m  eta={eta/60:5.1f}m",
              flush=True)

    raw = pd.DataFrame(all_rows, columns=["ticker", "date", "title"])
    print(f"[news] {len(raw):,} headlines collected across {raw['ticker'].nunique() if len(raw) else 0} tickers")

    if len(raw):
        grouped = raw.groupby(["ticker", "date"])["title"].agg(list).reset_index()
        grouped["n_headlines"] = grouped["title"].str.len().astype(int)
        grouped["headlines"] = grouped["title"].apply(
            lambda h: _truncate_to_tokens(" [SEP] ".join(h), tok, 512)
        )
        grouped = grouped[["ticker", "date", "headlines", "n_headlines"]]
    else:
        grouped = pd.DataFrame(columns=["ticker", "date", "headlines", "n_headlines"])

    full_idx = pd.MultiIndex.from_product(
        [tickers, trading_dates.normalize()], names=["ticker", "date"]
    ).to_frame(index=False)
    df = full_idx.merge(grouped, on=["ticker", "date"], how="left")
    df["headlines"]   = df["headlines"].fillna("NO_NEWS")
    df["n_headlines"] = df["n_headlines"].fillna(0).astype(int)
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_parquet(out_path, index=False)
    n_with_news = int((df["n_headlines"] > 0).sum())
    print(f"[news] saved {len(df):,} rows -> {out_path}  ({n_with_news:,} with >=1 headline)")

    _persist_to_drive(df)
    return df

trading_dates = pd.DatetimeIndex(sorted(prices_df["date"].dt.normalize().unique()))
news_df = download_news(STOCKS, DOWNLOAD_START, DOWNLOAD_END, trading_dates, NEWS_PATH)
news_df.head()

## 5. Data download: earnings dates (yfinance)

For each ticker, fetches `Ticker.earnings_dates` and tags `(ticker, date)` pairs that fall on an earnings announcement day. Missing/unavailable data defaults to `is_earnings_day=False`.

In [ ]:
EARNINGS_PATH = os.path.join(DATA_DIR, "earnings.parquet")

def download_earnings(tickers: List[str], trading_dates: pd.DatetimeIndex, out_path: str) -> pd.DataFrame:
    if os.path.exists(out_path):
        print(f"[earnings] cached -> {out_path}")
        return pd.read_parquet(out_path)

    print(f"[earnings] querying yfinance earnings_dates for {len(tickers)} tickers ...")
    earnings_lookup: Dict[str, set] = {}
    for t in tqdm(tickers, desc="earnings"):
        dates: set = set()
        try:
            tk = yf.Ticker(t)
            ed = tk.get_earnings_dates(limit=40)
            if ed is not None and len(ed) > 0:
                for d in ed.index:
                    try:
                        dates.add(pd.to_datetime(d).normalize())
                    except Exception:
                        continue
        except Exception:
            pass
        earnings_lookup[t] = dates

    rows = []
    for t in tickers:
        e_set = earnings_lookup.get(t, set())
        for d in trading_dates:
            rows.append({"ticker": t, "date": d.normalize(), "is_earnings_day": d.normalize() in e_set})
    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False)
    n_true = int(df["is_earnings_day"].sum())
    print(f"[earnings] saved {len(df):,} rows ({n_true:,} earnings flags) -> {out_path}")
    return df

earnings_df = download_earnings(STOCKS, trading_dates, EARNINGS_PATH)
earnings_df["is_earnings_day"].value_counts()

## 6. `MarketPulseDataset`

Joins prices + news + earnings on `(ticker, date)`. For each row, returns 5 days of OHLCV+return history, the day's headlines, the news count, the earnings flag, and the next-day directional label. Splits chronologically.

In [ ]:
def build_joined_frame(prices: pd.DataFrame, news: pd.DataFrame, earnings: pd.DataFrame) -> pd.DataFrame:
    p = prices.copy()
    p["date"] = pd.to_datetime(p["date"]).dt.normalize()
    n = news.copy()
    n["date"] = pd.to_datetime(n["date"]).dt.normalize()
    e = earnings.copy()
    e["date"] = pd.to_datetime(e["date"]).dt.normalize()

    df = p.merge(n, on=["ticker", "date"], how="left")
    df = df.merge(e, on=["ticker", "date"], how="left")
    df["headlines"] = df["headlines"].fillna("NO_NEWS")
    df["n_headlines"] = df["n_headlines"].fillna(0).astype(int)
    df["is_earnings_day"] = df["is_earnings_day"].fillna(False).astype(bool)

    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    df["next_return"] = df.groupby("ticker")["close"].shift(-1) - df["close"]
    df["label"] = (df["next_return"] > 0).astype(int)
    return df

class MarketPulseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, date_start: str, date_end: str, window: int = WINDOW):
        self.window = window
        self.feature_cols = PRICE_FEATURES
        self.records: List[dict] = []
        ds = pd.to_datetime(date_start).normalize()
        de = pd.to_datetime(date_end).normalize()

        for ticker, sub in df.groupby("ticker"):
            sub = sub.sort_values("date").reset_index(drop=True)
            feats = sub[self.feature_cols].to_numpy(dtype=np.float32)
            labels = sub["label"].to_numpy(dtype=np.int64)
            next_ret = sub["next_return"].to_numpy(dtype=np.float32)
            for i in range(window - 1, len(sub) - 1):
                d = sub["date"].iloc[i]
                if d < ds or d > de:
                    continue
                self.records.append({
                    "ticker": ticker,
                    "date": d,
                    "price_seq": feats[i - window + 1 : i + 1],
                    "headlines": sub["headlines"].iloc[i],
                    "n_headlines": int(sub["n_headlines"].iloc[i]),
                    "is_earnings_day": bool(sub["is_earnings_day"].iloc[i]),
                    "label": int(labels[i]),
                    "next_return": float(next_ret[i]),
                })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "price_seq": torch.from_numpy(r["price_seq"]).float(),
            "headlines": r["headlines"],
            "n_headlines": r["n_headlines"],
            "is_earnings_day": r["is_earnings_day"],
            "label": r["label"],
            "next_return": r["next_return"],
        }

def mp_collate(batch):
    return {
        "price_seq":      torch.stack([b["price_seq"] for b in batch], dim=0),
        "headlines":      [b["headlines"] for b in batch],
        "n_headlines":    torch.tensor([b["n_headlines"] for b in batch], dtype=torch.long),
        "is_earnings_day":torch.tensor([b["is_earnings_day"] for b in batch], dtype=torch.bool),
        "label":          torch.tensor([b["label"] for b in batch], dtype=torch.float32),
        "next_return":    torch.tensor([b["next_return"] for b in batch], dtype=torch.float32),
    }

joined_df = build_joined_frame(prices_df, news_df, earnings_df)
train_ds = MarketPulseDataset(joined_df, DATE_TRAIN_START, DATE_TRAIN_END)
val_ds   = MarketPulseDataset(joined_df, DATE_VAL_START,   DATE_VAL_END)
test_ds  = MarketPulseDataset(joined_df, DATE_TEST_START,  DATE_TEST_END)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=mp_collate, num_workers=0, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=mp_collate, num_workers=0, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=mp_collate, num_workers=0, drop_last=False)

print(f"train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

## 7. `FinBERTEncoder`

Loads `ProsusAI/finbert`, freezes everything except the **last 2 transformer blocks** and the **pooler**. For `"NO_NEWS"` rows it bypasses the transformer and substitutes a learned `no_news_embedding` parameter (initialized to zeros) so the no-news case stays differentiable.

In [ ]:
from transformers import AutoModel

class FinBERTEncoder(nn.Module):
    def __init__(self, model_name: str = "ProsusAI/finbert", unfreeze_last_n: int = 2):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size
        self.no_news_embedding = nn.Parameter(torch.zeros(self.hidden_size))

        for p in self.bert.parameters():
            p.requires_grad = False
        layers = self.bert.encoder.layer
        for layer in layers[-unfreeze_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True
        if hasattr(self.bert, "pooler") and self.bert.pooler is not None:
            for p in self.bert.pooler.parameters():
                p.requires_grad = True

    def forward(self, headlines: List[str]) -> torch.Tensor:
        device = self.no_news_embedding.device
        bsz = len(headlines)
        out = torch.empty(bsz, self.hidden_size, device=device)

        no_news_mask = [h == "NO_NEWS" for h in headlines]
        real_idx = [i for i, m in enumerate(no_news_mask) if not m]
        no_idx   = [i for i, m in enumerate(no_news_mask) if m]

        if no_idx:
            out[no_idx] = self.no_news_embedding.unsqueeze(0).expand(len(no_idx), -1)

        if real_idx:
            real_texts = [headlines[i] for i in real_idx]
            tok = self.tokenizer(real_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
            tok = {k: v.to(device) for k, v in tok.items()}
            outputs = self.bert(**tok)
            cls = outputs.last_hidden_state[:, 0, :]
            out[real_idx] = cls
        return out

## 8. `TCNEncoder`

4-layer Temporal Convolutional Network with dilations `[1, 2, 4, 8]`, kernel size 3, hidden 64, causal padding, residual connections. Input `(B, 6, 5)`, output `(B, 128)`.

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=0, dilation=dilation)

    def forward(self, x):
        x = F.pad(x, (self.pad, 0))
        return self.conv(x)

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        self.conv = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.res = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.act(out)
        out = self.drop(out)
        return out + self.res(x)

class TCNEncoder(nn.Module):
    def __init__(self, in_features=6, hidden=TCN_CHANNELS, out_dim=128, num_layers=TCN_LAYERS, kernel_size=3, dropout=0.2):
        super().__init__()
        dilations = [2 ** i for i in range(num_layers)]
        layers = []
        ch_in = in_features
        for d in dilations:
            layers.append(TemporalBlock(ch_in, hidden, kernel_size, d, dropout))
            ch_in = hidden
        self.tcn = nn.Sequential(*layers)
        self.proj = nn.Linear(hidden, out_dim)
        self.out_dim = out_dim

    def forward(self, x):
        x = x.transpose(1, 2)
        h = self.tcn(x)
        h = h.mean(dim=2)
        return self.proj(h)

## 9. Fusion: `MarketPulse` (gated) + `MarketPulseNoGate`

Both project text (768) and price (128) embeddings to 256 dims. The gated variant computes `g = sigmoid(W @ concat(text, price))` and outputs `g * text + (1 - g) * price`. The NoGate ablation simply concatenates and feeds 512 dims through the head.

In [ ]:
class MarketPulse(nn.Module):
    def __init__(self, finbert_encoder: FinBERTEncoder, tcn_encoder: TCNEncoder,
                 embed_dim: int = EMBED_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.finbert = finbert_encoder
        self.tcn     = tcn_encoder
        self.text_proj  = nn.Linear(self.finbert.hidden_size, embed_dim)
        self.price_proj = nn.Linear(self.tcn.out_dim,         embed_dim)
        self.gate = nn.Linear(2 * embed_dim, 1)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, price_seq, headlines):
        text_emb  = self.finbert(headlines)
        price_emb = self.tcn(price_seq)
        t = self.text_proj(text_emb)
        p = self.price_proj(price_emb)
        g = torch.sigmoid(self.gate(torch.cat([t, p], dim=-1)))
        fused = g * t + (1 - g) * p
        prob = self.head(fused).squeeze(-1)
        return prob, g.squeeze(-1)

class MarketPulseNoGate(nn.Module):
    def __init__(self, finbert_encoder: FinBERTEncoder, tcn_encoder: TCNEncoder,
                 embed_dim: int = EMBED_DIM, dropout: float = DROPOUT):
        super().__init__()
        self.finbert = finbert_encoder
        self.tcn     = tcn_encoder
        self.text_proj  = nn.Linear(self.finbert.hidden_size, embed_dim)
        self.price_proj = nn.Linear(self.tcn.out_dim,         embed_dim)
        self.head = nn.Sequential(
            nn.Linear(2 * embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, price_seq, headlines):
        text_emb  = self.finbert(headlines)
        price_emb = self.tcn(price_seq)
        t = self.text_proj(text_emb)
        p = self.price_proj(price_emb)
        prob = self.head(torch.cat([t, p], dim=-1)).squeeze(-1)
        return prob, None

## 10. Baselines

- **TFIDFLogistic**: TF-IDF on headlines + sklearn `LogisticRegression`.
- **FinBERTOnly**: FinBERT encoder + linear head.
- **TCNOnly**: TCN encoder + linear head.

In [ ]:
class TFIDFLogistic:
    def __init__(self, max_features: int = 10000):
        self.vec = TfidfVectorizer(max_features=max_features)
        self.clf = LogisticRegression(max_iter=1000)

    def fit(self, headlines: List[str], labels: List[int]):
        X = self.vec.fit_transform(headlines)
        self.clf.fit(X, labels)
        return self

    def predict_proba(self, headlines: List[str]) -> np.ndarray:
        X = self.vec.transform(headlines)
        return self.clf.predict_proba(X)[:, 1]

    def predict(self, headlines: List[str]) -> np.ndarray:
        return (self.predict_proba(headlines) > 0.5).astype(int)


class FinBERTOnly(nn.Module):
    def __init__(self, finbert_encoder: FinBERTEncoder):
        super().__init__()
        self.finbert = finbert_encoder
        self.head = nn.Sequential(nn.Linear(self.finbert.hidden_size, 1), nn.Sigmoid())

    def forward(self, price_seq, headlines):
        emb = self.finbert(headlines)
        prob = self.head(emb).squeeze(-1)
        return prob, None


class TCNOnly(nn.Module):
    def __init__(self, tcn_encoder: TCNEncoder):
        super().__init__()
        self.tcn = tcn_encoder
        self.head = nn.Sequential(nn.Linear(self.tcn.out_dim, 1), nn.Sigmoid())

    def forward(self, price_seq, headlines=None):
        emb = self.tcn(price_seq)
        prob = self.head(emb).squeeze(-1)
        return prob, None

## 11. Training

Trains all 5 models sequentially. Torch models share a single training loop with `AdamW` (FinBERT params at `2e-5`, others at `1e-4`), `CosineAnnealingLR`, BCE loss, early stopping (patience=3), and a sanity-check warning if val accuracy lands more than 3% below the spec target.

In [ ]:
METRICS_PATH = os.path.join(RESULTS_DIR, "metrics.json")

def _load_metrics() -> dict:
    if os.path.exists(METRICS_PATH):
        with open(METRICS_PATH, "r") as f:
            return json.load(f)
    return {}

def _save_metrics(d: dict):
    with open(METRICS_PATH, "w") as f:
        json.dump(d, f, indent=2, default=str)

def make_param_groups(model: nn.Module, lr_finbert: float, lr_other: float, weight_decay: float):
    finbert_params, other_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "bert." in name or "finbert.bert" in name or name.startswith("finbert.bert"):
            finbert_params.append(p)
        else:
            other_params.append(p)
    groups = []
    if finbert_params:
        groups.append({"params": finbert_params, "lr": lr_finbert})
    if other_params:
        groups.append({"params": other_params,   "lr": lr_other})
    return groups, weight_decay

@torch.no_grad()
def evaluate_torch(model: nn.Module, loader: DataLoader) -> Tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    all_preds, all_labels = [], []
    bce = nn.BCELoss(reduction="sum")
    for batch in loader:
        price_seq = batch["price_seq"].to(DEVICE)
        labels    = batch["label"].to(DEVICE)
        headlines = batch["headlines"]
        prob, _ = model(price_seq, headlines)
        prob = prob.clamp(1e-7, 1 - 1e-7)
        loss = bce(prob, labels)
        total_loss += loss.item()
        preds = (prob > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total   += labels.size(0)
        all_preds.append(prob.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
    if total == 0:
        return float("nan"), float("nan"), np.array([]), np.array([])
    return total_loss / total, correct / total, np.concatenate(all_preds), np.concatenate(all_labels)

def train_torch_model(model: nn.Module, name: str,
                      train_loader: DataLoader, val_loader: DataLoader,
                      max_epochs: int = MAX_EPOCHS, patience: int = EARLY_STOPPING_PATIENCE) -> dict:
    print(f"\n{'='*70}\nTraining {name}\n{'='*70}")
    model.to(DEVICE)
    groups, wd = make_param_groups(model, LR_FINBERT, LR_TCN, WEIGHT_DECAY)
    if not groups:
        print(f"[{name}] no trainable params \u2014 skipping training.")
        return {"name": name, "best_val_acc": float("nan"), "history": []}
    optim = torch.optim.AdamW(groups, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=max_epochs)
    bce = nn.BCELoss()

    best_val_acc, best_state, since_improve = -1.0, None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss, running_n = 0.0, 0
        for batch in tqdm(train_loader, desc=f"{name} ep{epoch}", leave=False):
            price_seq = batch["price_seq"].to(DEVICE)
            labels    = batch["label"].to(DEVICE)
            headlines = batch["headlines"]
            optim.zero_grad()
            prob, _ = model(price_seq, headlines)
            prob = prob.clamp(1e-7, 1 - 1e-7)
            loss = bce(prob, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            running_loss += loss.item() * labels.size(0)
            running_n    += labels.size(0)
        sched.step()
        train_loss = running_loss / max(running_n, 1)
        val_loss, val_acc, _, _ = evaluate_torch(model, val_loader)
        print(f"  epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            since_improve = 0
        else:
            since_improve += 1
            if since_improve >= patience:
                print(f"  early stop at epoch {epoch} (best val_acc={best_val_acc:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"{name}_best.pt")
        torch.save(best_state, ckpt_path)
        print(f"  saved checkpoint -> {ckpt_path}")

    expected = EXPECTED_ACCURACY.get(name)
    if expected is not None and best_val_acc < expected - 0.03:
        print(f"  [WARN] {name} val_acc={best_val_acc:.4f} is >3% below expected {expected:.2f} \u2014 check for bugs.")

    return {"name": name, "best_val_acc": best_val_acc, "history": history}

In [ ]:
def make_finbert():
    return FinBERTEncoder()

def make_tcn():
    return TCNEncoder(in_features=len(PRICE_FEATURES), hidden=TCN_CHANNELS,
                      out_dim=128, num_layers=TCN_LAYERS, kernel_size=3, dropout=0.2)

metrics_all = _load_metrics()

torch_models_to_train = [
    ("marketpulse",        lambda: MarketPulse(make_finbert(), make_tcn())),
    ("marketpulse_nogate", lambda: MarketPulseNoGate(make_finbert(), make_tcn())),
    ("finbert_only",       lambda: FinBERTOnly(make_finbert())),
    ("tcn_only",           lambda: TCNOnly(make_tcn())),
]

train_summary = {}
for name, ctor in torch_models_to_train:
    model = ctor()
    info = train_torch_model(model, name, train_loader, val_loader)
    train_summary[name] = info
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n--- Training TFIDFLogistic baseline ---")
tfidf = TFIDFLogistic()
train_headlines = [r["headlines"] for r in train_ds.records]
train_labels    = [r["label"]     for r in train_ds.records]
val_headlines   = [r["headlines"] for r in val_ds.records]
val_labels      = [r["label"]     for r in val_ds.records]
tfidf.fit(train_headlines, train_labels)
val_preds_tfidf = tfidf.predict(val_headlines)
val_acc_tfidf = float(accuracy_score(val_labels, val_preds_tfidf))
print(f"  tfidf_lr val_acc = {val_acc_tfidf:.4f}")
expected = EXPECTED_ACCURACY["tfidf_lr"]
if val_acc_tfidf < expected - 0.03:
    print(f"  [WARN] tfidf_lr val_acc={val_acc_tfidf:.4f} is >3% below expected {expected:.2f}.")
train_summary["tfidf_lr"] = {"name": "tfidf_lr", "best_val_acc": val_acc_tfidf, "history": []}

import pickle
with open(os.path.join(CHECKPOINT_DIR, "tfidf_lr_best.pkl"), "wb") as f:
    pickle.dump(tfidf, f)

metrics_all["train"] = {k: {"best_val_acc": v["best_val_acc"], "history": v["history"]} for k, v in train_summary.items()}
_save_metrics(metrics_all)
print(f"\nTraining done. Summary saved to {METRICS_PATH}")

## 12. Evaluation

Loads each best checkpoint, runs on the test set, and computes:
1. Directional accuracy (overall + stratified by news volume + stratified by earnings days).
2. Matthews Correlation Coefficient.
3. Long-short Sharpe ratio (annualized).

In [ ]:
@torch.no_grad()
def predict_torch_full(model: nn.Module, loader: DataLoader, return_gate: bool = False):
    model.eval()
    probs, labels, n_news_arr, earnings_arr, gates, next_rets = [], [], [], [], [], []
    for batch in loader:
        price_seq = batch["price_seq"].to(DEVICE)
        prob, gate = model(price_seq, batch["headlines"])
        probs.append(prob.detach().cpu().numpy())
        labels.append(batch["label"].numpy())
        n_news_arr.append(batch["n_headlines"].numpy())
        earnings_arr.append(batch["is_earnings_day"].numpy())
        next_rets.append(batch["next_return"].numpy())
        if return_gate and gate is not None:
            gates.append(gate.detach().cpu().numpy())
        else:
            gates.append(np.full((batch["label"].numel(),), np.nan))
    return (np.concatenate(probs), np.concatenate(labels), np.concatenate(n_news_arr),
            np.concatenate(earnings_arr), np.concatenate(gates), np.concatenate(next_rets))

def compute_metrics(probs: np.ndarray, labels: np.ndarray, n_news: np.ndarray,
                    earnings: np.ndarray, next_rets: np.ndarray) -> dict:
    preds = (probs > 0.5).astype(int)
    overall_acc = float(accuracy_score(labels, preds))
    high_mask = n_news >= HIGH_NEWS_THRESHOLD
    low_mask  = ~high_mask
    earn_mask = earnings.astype(bool)
    nonearn_mask = ~earn_mask
    high_acc = float(accuracy_score(labels[high_mask], preds[high_mask])) if high_mask.any() else float("nan")
    low_acc  = float(accuracy_score(labels[low_mask],  preds[low_mask]))  if low_mask.any()  else float("nan")
    earn_acc = float(accuracy_score(labels[earn_mask], preds[earn_mask])) if earn_mask.any() else float("nan")
    nonearn_acc = float(accuracy_score(labels[nonearn_mask], preds[nonearn_mask])) if nonearn_mask.any() else float("nan")
    mcc = float(matthews_corrcoef(labels, preds))
    pnl = (2 * preds - 1) * next_rets
    if pnl.std() > 0:
        sharpe = float(pnl.mean() / pnl.std() * math.sqrt(252))
    else:
        sharpe = 0.0
    return {
        "accuracy":         overall_acc,
        "accuracy_high_news": high_acc,
        "accuracy_low_news":  low_acc,
        "accuracy_earnings":     earn_acc,
        "accuracy_non_earnings": nonearn_acc,
        "mcc":              mcc,
        "sharpe":           sharpe,
        "n_high_news_days": int(high_mask.sum()),
        "n_earnings_days":  int(earn_mask.sum()),
        "n_total":          int(len(labels)),
    }

def load_torch_ckpt(name: str, ctor) -> Optional[nn.Module]:
    path = os.path.join(CHECKPOINT_DIR, f"{name}_best.pt")
    if not os.path.exists(path):
        print(f"[eval] checkpoint missing for {name} \u2014 skipping.")
        return None
    model = ctor()
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    return model

eval_results = {}
torch_eval_specs = [
    ("marketpulse",        lambda: MarketPulse(make_finbert(), make_tcn()),        True),
    ("marketpulse_nogate", lambda: MarketPulseNoGate(make_finbert(), make_tcn()),  False),
    ("finbert_only",       lambda: FinBERTOnly(make_finbert()),                    False),
    ("tcn_only",           lambda: TCNOnly(make_tcn()),                            False),
]

gate_payload = None
for name, ctor, has_gate in torch_eval_specs:
    model = load_torch_ckpt(name, ctor)
    if model is None:
        continue
    probs, labels, n_news, earn, gates, next_rets = predict_torch_full(model, test_loader, return_gate=has_gate)
    m = compute_metrics(probs, labels, n_news, earn, next_rets)
    eval_results[name] = m
    if has_gate:
        gate_payload = {"probs": probs, "labels": labels, "gates": gates,
                        "n_news": n_news, "earnings": earn, "next_rets": next_rets}
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tfidf_path = os.path.join(CHECKPOINT_DIR, "tfidf_lr_best.pkl")
if os.path.exists(tfidf_path):
    with open(tfidf_path, "rb") as f:
        tfidf = pickle.load(f)
    test_headlines = [r["headlines"]   for r in test_ds.records]
    test_labels    = np.array([r["label"]       for r in test_ds.records])
    test_n_news    = np.array([r["n_headlines"] for r in test_ds.records])
    test_earn      = np.array([r["is_earnings_day"] for r in test_ds.records])
    test_next_rets = np.array([r["next_return"] for r in test_ds.records])
    tfidf_probs    = tfidf.predict_proba(test_headlines)
    eval_results["tfidf_lr"] = compute_metrics(tfidf_probs, test_labels, test_n_news, test_earn, test_next_rets)

print(f"\n{'Model':<22} {'Acc':>7} {'HighN':>7} {'Earn':>7} {'MCC':>7} {'Sharpe':>8}")
print("-" * 64)
for name, m in eval_results.items():
    print(f"{name:<22} {m['accuracy']:>7.4f} {m['accuracy_high_news']:>7.4f} "
          f"{m['accuracy_earnings']:>7.4f} {m['mcc']:>7.4f} {m['sharpe']:>8.3f}")

metrics_all = _load_metrics()
metrics_all["test"] = eval_results
_save_metrics(metrics_all)
print(f"\nTest metrics saved to {METRICS_PATH}")

## 13. Visualizations

5 plots saved to `results/figures/`:
1. Attention gate distribution (high-news vs low-news vs earnings).
2. Accuracy bar chart across all 5 models.
3. Calibration curve for MarketPulse.
4. SHAP per-lag importance for the TCN encoder.
5. Metrics summary table (`.png` + `.csv`).

In [ ]:
if gate_payload is not None and not np.all(np.isnan(gate_payload["gates"])):
    g = gate_payload["gates"]
    n_news = gate_payload["n_news"]
    earn   = gate_payload["earnings"].astype(bool)
    high_mask = n_news >= HIGH_NEWS_THRESHOLD
    low_mask  = ~high_mask

    fig, ax = plt.subplots(figsize=(8, 5))
    bins = np.linspace(0, 1, 30)
    if high_mask.any():
        ax.hist(g[high_mask], bins=bins, alpha=0.55, label=f"High-news (n>={HIGH_NEWS_THRESHOLD})", color="tab:blue")
    if low_mask.any():
        ax.hist(g[low_mask],  bins=bins, alpha=0.55, label="Low-news",  color="tab:orange")
    if earn.any():
        ax.hist(g[earn],      bins=bins, alpha=0.55, label="Earnings days", color="tab:red")
    ax.set_xlabel("Gate value g (text weight)")
    ax.set_ylabel("Count")
    ax.set_title("MarketPulse attention gate distribution")
    ax.legend()
    fig.tight_layout()
    out = os.path.join(FIGURES_DIR, "attention_gate_distribution.png")
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")
else:
    print("[viz] no gate payload available \u2014 skipping attention_gate_distribution.png")

In [ ]:
model_order = ["tfidf_lr", "tcn_only", "finbert_only", "marketpulse_nogate", "marketpulse"]
model_labels = ["TF-IDF LR", "TCN-Only", "FinBERT-Only", "MarketPulse-NoGate", "MarketPulse"]

overall  = [eval_results.get(m, {}).get("accuracy",          float("nan")) for m in model_order]
high_acc = [eval_results.get(m, {}).get("accuracy_high_news",float("nan")) for m in model_order]
earn_acc = [eval_results.get(m, {}).get("accuracy_earnings", float("nan")) for m in model_order]

x = np.arange(len(model_order))
w = 0.27
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(x - w, overall,  width=w, label="Overall")
ax.bar(x,     high_acc, width=w, label="High-news")
ax.bar(x + w, earn_acc, width=w, label="Earnings days")
ax.axhline(0.60, color="black", linestyle="--", linewidth=1, label="0.60 target")
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=15)
ax.set_ylabel("Directional accuracy")
ax.set_ylim(0.40, max(0.70, max([v for v in overall + high_acc + earn_acc if not np.isnan(v)] + [0.65]) + 0.03))
ax.set_title("Directional accuracy by model")
ax.legend()
fig.tight_layout()
out = os.path.join(FIGURES_DIR, "accuracy_bar_chart.png")
fig.savefig(out, dpi=150)
plt.show()
print(f"saved {out}")

In [ ]:
if gate_payload is not None:
    probs = gate_payload["probs"]
    labels = gate_payload["labels"]
    frac_pos, mean_pred = calibration_curve(labels, probs, n_bins=10, strategy="uniform")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "o-", label="MarketPulse")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title("Calibration curve (MarketPulse)")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    out = os.path.join(FIGURES_DIR, "calibration_curve.png")
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")
else:
    print("[viz] no gate payload \u2014 skipping calibration_curve.png")

In [ ]:
try:
    import shap
    tcn_model = load_torch_ckpt("tcn_only", lambda: TCNOnly(make_tcn()))
    if tcn_model is None:
        raise RuntimeError("no tcn_only checkpoint")

    bg_n = min(64, len(train_ds))
    test_n = min(128, len(test_ds))
    bg = torch.stack([train_ds[i]["price_seq"] for i in range(bg_n)]).to(DEVICE)
    samp = torch.stack([test_ds[i]["price_seq"] for i in range(test_n)]).to(DEVICE)

    class _TCNWrap(nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m
        def forward(self, x):
            prob, _ = self.m(x)
            return prob.unsqueeze(-1)

    wrapped = _TCNWrap(tcn_model).to(DEVICE).eval()
    explainer = shap.GradientExplainer(wrapped, bg)
    shap_vals = explainer.shap_values(samp)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]
    sv = np.asarray(shap_vals)
    if sv.ndim == 4 and sv.shape[-1] == 1:
        sv = sv.squeeze(-1)
    abs_per_lag_feat = np.mean(np.abs(sv), axis=0)
    if abs_per_lag_feat.shape[0] == len(PRICE_FEATURES):
        abs_per_lag_feat = abs_per_lag_feat.T

    lag_labels = [f"day -{WINDOW - i}" for i in range(WINDOW)]
    fig, ax = plt.subplots(figsize=(8, 5))
    bottoms = np.zeros(WINDOW)
    for fi, fname in enumerate(PRICE_FEATURES):
        vals = abs_per_lag_feat[:, fi]
        ax.bar(lag_labels, vals, bottom=bottoms, label=fname)
        bottoms = bottoms + vals
    ax.set_ylabel("Mean |SHAP value|")
    ax.set_title("TCN price-encoder feature importance per lag")
    ax.legend(loc="upper left", ncol=2, fontsize=9)
    fig.tight_layout()
    out = os.path.join(FIGURES_DIR, "shap_price_lags.png")
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")
    del tcn_model, wrapped
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as e:
    print(f"[viz] SHAP plot failed: {e}")

In [ ]:
cols = ["Model", "Accuracy", "MCC", "Sharpe", "High-News Acc", "Earnings Acc"]
rows = []
for m, label in zip(model_order, model_labels):
    r = eval_results.get(m, {})
    rows.append([
        label,
        f"{r.get('accuracy', float('nan')):.4f}",
        f"{r.get('mcc',      float('nan')):.4f}",
        f"{r.get('sharpe',   float('nan')):.3f}",
        f"{r.get('accuracy_high_news', float('nan')):.4f}",
        f"{r.get('accuracy_earnings',  float('nan')):.4f}",
    ])
summary_df = pd.DataFrame(rows, columns=cols)
csv_out = os.path.join(FIGURES_DIR, "metrics_summary_table.csv")
summary_df.to_csv(csv_out, index=False)

fig, ax = plt.subplots(figsize=(11, 0.6 + 0.5 * len(rows)))
ax.axis("off")
tbl = ax.table(cellText=rows, colLabels=cols, cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.4)
mp_idx = model_labels.index("MarketPulse") + 1
for c in range(len(cols)):
    cell = tbl[(mp_idx, c)]
    cell.set_text_props(weight="bold")
ax.set_title("MarketPulse: model comparison on test set", pad=12)
fig.tight_layout()
png_out = os.path.join(FIGURES_DIR, "metrics_summary_table.png")
fig.savefig(png_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {png_out}")
print(f"saved {csv_out}")
print("\nDone. All artifacts in:", RESULTS_DIR)